In [ ]:
print(1)
#
# - 1 - database
# - 2 - connection to telethon, session
# - 3 - get chats
# - 4 - yaml config
# - 5 - get messages
from dotenv import load_dotenv

load_dotenv()

# Part 1 - Database

In [ ]:
# option 1 - mongodb pymongo client
from pymongo import MongoClient
import os
from loguru import logger

logger.info("Starting MongoDB setup...")

# Get MongoDB connection string and database name from environment variables
conn_str = os.getenv("MONGO_CONN_STR")
db_name = os.getenv("MONGO_DB_NAME", "telegram-messages-dec-2024")

logger.info(f"Using database name: {db_name}")
logger.info("Attempting to connect to MongoDB...")

# Connect to MongoDB
client = MongoClient(conn_str)
logger.info("Successfully connected to MongoDB")

# MongoDB creates databases and collections automatically when you first store data
# But we can explicitly create them to ensure they exist
logger.info("Checking if database exists...")
if db_name not in client.list_database_names():
    logger.info(f"Creating database: {db_name}")
    db = client[db_name]
else:
    logger.info(f"Using existing database: {db_name}")
    db = client[db_name]

# Define collections we'll need
collections = {
    "messages": "telegram_messages",
    "chats": "telegram_chats",
    "users": "telegram_users",
    "heartbeats": "telegram_heartbeats"
}

logger.info("Starting collection setup...")

# Create collections if they don't exist
for purpose, collection_name in collections.items():
    logger.info(f"Checking collection: {collection_name}")
    if collection_name not in db.list_collection_names():
        logger.info(f"Creating collection: {collection_name}")
        db.create_collection(collection_name)
    else:
        logger.info(f"Using existing collection: {collection_name}")

logger.info("Collection setup complete")

# add item, read items - to the test heartbeats collection
logger.info("Testing heartbeats collection...")
heartbeats_collection = db.heartbeats
from datetime import datetime

logger.info("Inserting test heartbeat...")
heartbeats_collection.insert_one({"timestamp": datetime.now()})
logger.info("Reading test heartbeat...")
heartbeats_collection.find_one()
logger.info("MongoDB setup complete")


In [ ]:
client, db

# Part 2 - Connection to Telethon

In [ ]:
# 1 - general "get_telethon_client" function
# trajectory 1: save and load conn from disk
# 2 - create new conn from scratch -> save conn to disk
# 3 - check if conn is present on disk
# 4 - load conn from disk
# trajectoty 2: save conn to db
# 5 - create new conn from scratch -> save conn to db
# 6 - load conn from db
# 7 - check if conn is present in db


Trajectory 1: Save and load connection from disk

In [ ]:
from enum import Enum


class StorageMode(str, Enum):
    TO_DATABASE = "to_database"
    TO_DISK = "to_disk"


class TelethonClientManager:
    def __init__(self, storage_mode: StorageMode):
        self.storage_mode = storage_mode

    def get_telethon_client(self, user_id):
        if self.storage_mode == StorageMode.TO_DISK:
            return _get_telethon_client_from_disk(user_id)
        elif self.storage_mode == StorageMode.TO_DATABASE:
            return _get_telethon_client_from_database(user_id)

    # region trajectory 1 save and load conn from disk
    def _get_telethon_client_from_disk(self, user_id):

        if self._check_if_conn_is_present_on_disk(user_id):
            return self._load_conn_from_disk(user_id)
        else:
            return self._create_new_telethon_client_and_save_to_disk(user_id)

    # 2 - create new conn from scratch -> save conn to disk
    def _create_new_telethon_client_and_save_to_disk(self, user_id):
        pass

    # 3 - check if conn is present on disk
    def _check_if_conn_is_present_on_disk(self, user_id):
        pass

    # 4 - load conn from disk
    def _load_conn_from_disk(self, user_id):
        pass

    # endregion trajectory 1

    # region trajectory 2 save conn to db
    def _get_telethon_client_from_database(self, user_id):
        raise NotImplementedError

    # 5 - create new conn from scratch -> save conn to db
    def _create_new_telethon_client_and_save_to_db(self, user_id):
        pass

    # 6 - load conn from db
    def _load_conn_from_db(self, user_id):
        pass

    # 7 - check if conn is present in db
    def _check_if_conn_is_present_in_db(self, user_id):
        pass

    # endregion trajectory 2

Trajectory 2: Create new connection from scratch -> save connection to disk

# Part 3 - Get Chats

In [ ]:
# idea 1: just load chat list
# idea 2: cache chat list. Save timestamp. If timestamp is older than 1 day, load chat list.
# idea 3: add --refresh flag. If flag is set, load chat list from scratch.
# idea 4: calculate fancy stats about chats.
# Basic:
# private messages
# group chats
# channels
# bots

# Advanced:
# owned groups
# owned bots
# owned channels

# big groups (> 1000)
# big channels

# Bonus: size / time-based stats
# for each of the above - count which have recent messages (last 30 days)

import asyncio
from typing import List
from telethon.tl.custom import Dialog

# class ChatCache(BaseModel):
#     timestamp: datetime
#     chats: List[Dialog]

# async def save_chat_cache(chats: List[Dialog], cache_file: Path = Path("data/chat_cache.json")):
#     """Save chat list to cache file with timestamp"""
#     cache_data = ChatCache(
#         timestamp=datetime.now(),
#         chats=chats
#     )
#
#     cache_file.parent.mkdir(exist_ok=True)
#     cache_file.write_text(cache_data.model_dump_json(indent=2))
#     logger.debug(f"Saved chat cache to {cache_file}")
import nest_asyncio

nest_asyncio.apply()


# Load our extension
# %load_ext telethon_jupyter

async def get_chat_list() -> List[Dialog]:
    client = await get_telethon_client()
    chats = await client.get_dialogs()
    return chats


# Store chats in memory
chats = asyncio.run(get_chat_list())

# Now you can work with them directly:
print("Total chats:", len(chats))
print("\nExample chats:")
for chat in chats[:5]:  # Show first 5 chats
    print(chat)

# Filter and analyze:
groups = [chat for chat in chats if chat.is_group]
channels = [chat for chat in chats if chat.is_channel]
users = [chat for chat in chats if chat.is_user]

print(f"\nFound {len(groups)} groups, {len(channels)} channels, {len(users)} users")

# Find largest groups
largest_groups = sorted(
    [chat for chat in groups if hasattr(chat.entity, 'participants_count')],
    key=lambda x: getattr(x.entity, 'participants_count', 0),
    reverse=True
)[:5]

print("\nLargest groups:")
for chat in largest_groups:
    print(f"{chat.entity.title}: {chat.entity.participants_count} members")

In [ ]:
# Get one chat and explore its attributes
chat = chats[0]

print("Basic chat info:")
print(f"Title: {chat.title}")
print(f"ID: {chat.id}")
print(f"Type: {'channel' if chat.is_channel else 'group' if chat.is_group else 'user'}")
print(f"Unread: {chat.unread_count}")

# Look at the entity object
print("\nEntity attributes:")
for attr in dir(chat.entity):
    if not attr.startswith('_'):  # Skip private attributes
        print(f"{attr}: {getattr(chat.entity, attr)}")

In [ ]:
# Admin status
print("Admin status:")
print(f"creator: {chat.entity.creator}")
print(f"admin_rights: {chat.entity.admin_rights}")

# Size
print("\nSize:")
print(f"participants_count: {chat.entity.participants_count}")

# Type indicators
print("\nType indicators:")
print(f"broadcast: {chat.entity.broadcast}")
print(f"megagroup: {chat.entity.megagroup}")

# Features
print("\nFeatures:")
print(f"forum: {chat.entity.forum}")
print(f"username: {chat.entity.username}")
print(f"verified: {chat.entity.verified}")


In [ ]:
ALLOWED = False
# ALLOWED=True

if not ALLOWED:
    print(
        "Old code version with naive implementation. Category counts don't sum up to total chats. - reesarch about this below.")
else:

    # Basic counts
    private_chats = [c for c in chats if c.is_user and not getattr(c.entity, 'bot',
                                                                   False)]  # Fixed private chat detection
    groups = [c for c in chats if c.is_group]
    channels = [c for c in chats if c.is_channel]
    bots = [c for c in chats if c.is_user and getattr(c.entity, 'bot', True)]


    def get_creator(chat):
        try:
            return chat.entity.creator
        except:
            return None


    # Ownership counts
    owned_groups = [c for c in groups if get_creator(c)]
    owned_channels = [c for c in channels if get_creator(c)]
    owned_bots = [c for c in bots if get_creator(c)]  # Added bot ownership

    print("Chat Statistics:")
    print(f"Private chats: {len(private_chats)}")

    print(f"\nGroups: {len(groups)}")
    print(f"  - owned: {len(owned_groups)}")

    print(f"\nChannels: {len(channels)}")
    print(f"  - owned: {len(owned_channels)}")

    print(f"\nBots: {len(bots)}")
    print(f"  - owned: {len(owned_bots)}")

In [ ]:


ALLOWED = False
# ALLOWED=True

if not ALLOWED:
    print(
        "Deprecated research code, research finished. Findings: groups with message history enabled are marked as channels in telethon - need to work around that for correct stats.")
else:
    # Let's debug our categorization
    total_chats = len(chats)
    private_chats = [c for c in chats if
                     c.is_user and not getattr(c.entity, 'bot', False)]
    groups = [c for c in chats if c.is_group]
    channels = [c for c in chats if c.is_channel]
    bots = [c for c in chats if c.is_user and getattr(c.entity, 'bot', True)]

    # Verify our counts
    categorized = len(private_chats) + len(groups) + len(channels) + len(bots)

    print("Verification:")
    print(f"Total chats: {total_chats}")
    print(f"Sum of categories: {categorized}")
    print(f"Difference: {total_chats - categorized}")

    # Let's check if any chats are uncategorized or double-counted
    uncategorized = []
    for chat in chats:
        categories = []
        if chat.is_user and not getattr(chat.entity, 'bot', False):
            categories.append('private')
        if chat.is_group:
            categories.append('group')
        if chat.is_channel:
            categories.append('channel')
        if chat.is_user and getattr(chat.entity, 'bot', True):
            categories.append('bot')

        if len(categories) != 1:
            # print(f"\nChat '{chat.title}' belongs to categories: {categories}")
            pass
        elif 'group' in categories:
            print(f"Group chat: {chat.title}")

In [ ]:
# Fixed categorization
def is_true_channel(chat):
    return chat.is_channel and not chat.is_group


# Basic counts
private_chats = [c for c in chats if c.is_user and not getattr(c.entity, 'bot', False)]
groups = [c for c in chats if c.is_group]  # Keep this as is
channels = [c for c in chats if is_true_channel(c)]  # Only true channels
bots = [c for c in chats if c.is_user and getattr(c.entity, 'bot', True)]

# Ownership counts
owned_groups = [c for c in groups if get_creator(c)]
owned_channels = [c for c in channels if get_creator(c)]


def get_stats_message():
    stats_message = "Chat Statistics:\n"

    stats_message += f"Private chats: {len(private_chats)}\n"

    stats_message += f"\nGroups: {len(groups)}\n"
    stats_message += f"  - owned: {len(owned_groups)}\n"

    stats_message += f"\nChannels: {len(channels)}\n"
    stats_message += f"  - owned: {len(owned_channels)}\n"

    stats_message += f"\nBots: {len(bots)}\n"

    # Verification
    total = len(private_chats) + len(groups) + len(channels) + len(bots)
    stats_message += f"\nTotal chats: {len(chats)}\n"
    stats_message += f"Sum of categories: {total}\n"
    stats_message += f"Difference: {len(chats) - total}\n"

    return stats_message


print(get_stats_message())

In [ ]:
from random import choice


def get_basic_chat_stats(chat):
    stats = {
        "title": chat.title,
        "last_message_date": chat.date,
        "creation_date": getattr(chat.entity, 'date', None),
        "participants_count": getattr(chat.entity, 'participants_count', None),
    }
    return stats


# Sample from each category and show stats
print("=== Random Samples ===")

# Private chat
if private_chats:
    sample = choice(private_chats)
    print("\nPrivate Chat Sample:")
    stats = get_basic_chat_stats(sample)
    print(f"Title: {stats['title']}")
    print(f"Last message: {stats['last_message_date']}")
    print(f"Added: {stats['creation_date']}")
    # We might need to use client.get_entity() to get more user details

# Group
if groups:
    sample = choice(groups)
    print("\nGroup Sample:")
    stats = get_basic_chat_stats(sample)
    print(f"Title: {stats['title']}")
    print(f"Members: {stats['participants_count']}")
    print(f"Last message: {stats['last_message_date']}")
    print(f"Added: {stats['creation_date']}")
    # print(f"Is megagroup: {sample.entity.megagroup}")

# Channel
if channels:
    sample = choice(channels)
    print("\nChannel Sample:")
    stats = get_basic_chat_stats(sample)
    print(f"Title: {stats['title']}")
    print(f"Subscribers: {stats['participants_count']}")
    print(f"Last message: {stats['last_message_date']}")
    print(f"Added: {stats['creation_date']}")
    print(f"Broadcast: {sample.entity.broadcast}")

# Size distribution
print("\n=== Size Statistics ===")
big_groups = [g for g in groups if getattr(g.entity, 'participants_count', 0) > 1000]
big_channels = [c for c in channels if
                getattr(c.entity, 'participants_count', 0) > 1000]

print(f"Big groups (>1000): {len(big_groups)} out of {len(groups)}")
print(f"Big channels (>1000): {len(big_channels)} out of {len(channels)}")

In [ ]:
def plot_size_distribution(items, title):
    sizes = [getattr(item.entity, 'participants_count', 0) for item in items]
    max_size = max(sizes)

    # Custom bin edges for more intuitive intervals
    bins = [
        1, 3, 10, 30, 100, 300,  # Small groups/channels
        1000, 3000, 10000,  # Medium
        30000, 100000,  # Large
        300000, 1000000  # Huge
    ]

    # Filter out empty upper bins
    while bins[-1] > max_size * 1.1:  # Keep one empty bin for visual clarity
        bins.pop()

    plt.figure(figsize=(12, 6))

    # Calculate histogram data
    counts, edges = np.histogram(sizes, bins=bins)

    # Plot bars manually with fixed width in log space
    bar_width = 0.8  # Width in log space
    log_edges = np.log10(edges[:-1])

    plt.bar(edges[:-1], counts,
            width=[edge * bar_width for edge in edges[:-1]],
            # Width proportional to x position
            align='center')

    plt.xscale('log')

    # Create interval labels
    labels = [f'{bins[i]:,}-{bins[i + 1]:,}' for i in range(len(bins) - 1)]
    plt.xticks(bins[:-1], labels, rotation=45, ha='right')

    # Add value labels on top of bars
    for i, count in enumerate(counts):
        if count > 0:  # Only label non-empty bars
            plt.text(edges[i], count, f'{int(count)}',
                     va='bottom', ha='center')

    plt.title(f'{title} Size Distribution (n={len(items)})')
    plt.xlabel('Number of participants')
    plt.ylabel('Count')

    # Add grid for better readability
    plt.grid(True, alpha=0.3)

    # Adjust layout to prevent label cutoff
    plt.tight_layout()

    plt.show()

    # Print statistics
    print(f"\n{title} size statistics:")
    print(f"Median size: {np.median(sizes):,.0f}")
    print(f"Mean size: {np.mean(sizes):,.0f}")
    print(f"Max size: {max_size:,}")


# Plot distributions
plot_size_distribution(groups, 'Groups')
plot_size_distribution(channels, 'Channels')

In [ ]:
def plot_small_size_distribution(items, title):
    sizes = [getattr(item.entity, 'participants_count', 0) for item in items]

    # Custom bin edges for small groups
    bins = list(range(1, 12))  # 1 to 11 to capture sizes 1-10

    plt.figure(figsize=(10, 6))

    # Calculate histogram data
    counts, edges = np.histogram(sizes, bins=bins)

    # Plot bars
    plt.bar(range(1, 11), counts, width=0.7, align='center')

    # Set x-axis ticks to whole numbers
    plt.xticks(range(1, 11))

    # Add value labels on top of bars
    for i, count in enumerate(counts):
        if count > 0:  # Only label non-empty bars
            plt.text(i + 1, count, f'{int(count)}',
                     va='bottom', ha='center')

    plt.title(f'{title} Size Distribution (1-10 members, n={sum(counts)})')
    plt.xlabel('Number of participants')
    plt.ylabel('Count')

    # Add grid for better readability
    plt.grid(True, alpha=0.3)

    # Adjust layout
    plt.tight_layout()

    plt.show()


# Plot distributions
plot_small_size_distribution(groups, 'Groups')
plot_small_size_distribution(channels, 'Channels')

In [ ]:
chat = chats[0]

# for attr in dir(chat):
#     if not attr.startswith('_'):  # Skip private attributes
#         print(f"{attr}: {getattr(chat, attr)}")

data = chat.entity.to_json()
data

In [ ]:
from telethon.tl.types import Channel, Chat, User

# Let's analyze one chat
chat = chats[0]

print("=== Dialog attributes we use ===")
print(f"is_user: {chat.is_user}")
print(f"is_group: {chat.is_group}")
print(f"is_channel: {chat.is_channel}")

print("\n=== Entity equivalent checks ===")
entity = chat.entity
print(f"is_user: {isinstance(entity, User)}")
print(f"is_group: {isinstance(entity, Chat)}")
print(f"is_channel: {isinstance(entity, Channel)}")
print(
    f"is_channel but not megagroup: {isinstance(entity, Channel) and not entity.megagroup}")


# Let's try to rewrite our categorization using only entities
def categorize_entity(entity):
    if isinstance(entity, User):
        return 'private' if not getattr(entity, 'bot', False) else 'bot'
    elif isinstance(entity, Chat):
        return 'group'
    elif isinstance(entity, Channel):
        return 'channel' if not entity.megagroup else 'group'
    return 'unknown'


# Test the new categorization
print("\n=== Testing new categorization ===")
categories = {}
for chat in chats:  # Test first 5 chats
    old_cat = 'private' if chat.is_user else 'group' if chat.is_group else 'channel'
    new_cat = categorize_entity(chat.entity)
    if old_cat != new_cat:
        # print(f"Mismatch: {old_cat} vs {new_cat} for {chat.entity.title}")
        print(f"Mismatch: {old_cat} vs {new_cat}")
    # else:
    #     print(f"Match: {old_cat} vs {new_cat} for {chat.entity.title}")


In [ ]:
# Let's analyze one chat
chat = chats[0]

print("=== Dialog vs Entity categorization ===")


# Original Dialog-based categorization
def categorize_dialog(chat):
    if chat.is_user:
        return 'bot' if getattr(chat.entity, 'bot', False) else 'private'
    elif chat.is_group:
        return 'group'
    elif chat.is_channel:
        return 'channel'
    return 'unknown'


# New Entity-based categorization
def categorize_entity(entity):
    if isinstance(entity, User):
        return 'bot' if getattr(entity, 'bot', False) else 'private'
    elif isinstance(entity, Chat):
        return 'group'
    elif isinstance(entity, Channel):
        return 'channel' if not entity.megagroup else 'group'
    return 'unknown'


# Test the categorizations
print("\n=== Testing categorization (first 5 chats) ===")
for chat in chats[:5]:
    old_cat = categorize_dialog(chat)
    new_cat = categorize_entity(chat.entity)
    print(f"Title: {chat.entity.title}")
    print(f"Dialog cat: {old_cat}")
    print(f"Entity cat: {new_cat}")
    print()

# Count all categories
dialog_cats = {}
entity_cats = {}

for chat in chats:
    d_cat = categorize_dialog(chat)
    e_cat = categorize_entity(chat.entity)

    if d_cat != e_cat:
        print(f"Mismatch: {d_cat} vs {e_cat} for {chat.entity.title}")
        res = chat

    dialog_cats[d_cat] = dialog_cats.get(d_cat, 0) + 1
    entity_cats[e_cat] = entity_cats.get(e_cat, 0) + 1

print("=== Total counts ===")
print("Dialog categorization:", dialog_cats)
print("Entity categorization:", entity_cats)

In [ ]:
res.__dict__


In [ ]:
res.entity.to_json()

In [ ]:
# Plan:
# 1 - general get_chats
# 2 - _get_chats_from_disk
# 3 - _get_chats_from_client
# 4 - _has_fresh_chats_on_disk

In [ ]:


type_map = {
    'Channel': Channel,
    'Chat': Chat,
    'User': User
}


def _get_chats_from_disk():
    data_path = Path("data/chats.json")

    data = json.loads(data_path.read_text())
    chats = [
        type_map[entity['type']].from_json(entity['data'])
        for entity in data['entities']
    ]
    return chats


async def get_chat_list() -> List[Dialog]:
    client = await get_telethon_client()
    chats = await client.get_dialogs()
    return chats


def _get_chats_from_client():
    # Store chats in memory
    chats = asyncio.run(get_chat_list())
    return [chat.entity for chat in chats]


def _has_fresh_chats_on_disk():
    data_path = Path("data/chats.json")

    if not data_path.exists():
        return False

    data = json.loads(data_path.read_text())
    timestamp = data['timestamp']
    return datetime.now() - datetime.fromisoformat(timestamp) < timedelta(days=1)


def _save_chats_to_disk(chats):
    data = {
        'entities': {
            'type': str(type(chat)),
            'data': chat.to_json()
        },
        'timestamp': datetime.now().isoformat()
    }

    data_path = Path("data/chats.json")
    data_path.parent.mkdir(exist_ok=True)
    data_path.write_text(json.dumps(data))


def get_chats():
    if _has_fresh_chats_on_disk():
        return _get_chats_from_disk()
    else:
        chats = _get_chats_from_client()
        _save_chats_to_disk(chats)
        return chats



In [ ]:
# save chats to disk
# test step 1
chats = _get_chats_from_client()

In [ ]:
from pathlib import Path
import json

data_path = Path("data/chats.json")
data = json.loads(data_path.read_text())

# chats = [type_map[entity['type']](entity['data']) for entity in data['entities']]

In [ ]:
x = data['entities'][0]
t = type_map[x['type']]
d = json.loads(x['data'])
new_d = {
    'timestamp': data['timestamp'],
    'entities': [
        chat['data'] for chat in data['entities']
    ]
}
json.dump(new_d, open("data/chats.json", "w"), indent=2)

In [ ]:
for i in data['entities']:
    d = json.loads(i['data'])
    if "_" not in d:
        print(d)
    print(d['_'])

In [ ]:
str(type(channels[0].entity).__name__)

In [ ]:
dir(chat)

In [ ]:
chat.date
# chat.last_message_date
chat = groups[0]
chat.date

chat.entity.date

# Part 4

In [ ]:
from pathlib import Path
import asyncio
from loguru import logger
from telethon_manager import StorageMode
from dotenv import load_dotenv
from datetime import datetime, timedelta
import json
from get_chat_list import get_chats
from main import get_telethon_client
from download_messages_draft import get_messages_sample, save_messages, calculate_stats
import random

load_dotenv()
# setup_logger(logger, level="DEBUG" if debug else "INFO")

client = await get_telethon_client()
# if not client:
#     logger.error("Failed to get client")
#     return

chats = await get_chats()
logger.info(f"Found {len(chats)} chats")

# let's select a random sub-sample of chats
random.shuffle(chats)
chats = chats[:20]

messages = await get_messages_sample(client,
                                     chats)  # Start with just 5 chats for testing
logger.info(f"Downloaded messages from {len(messages)} chats")

save_messages(messages)


In [ ]:

stats = calculate_stats(messages)
logger.info("Message statistics:")
for key, value in stats.items():
    logger.info(f"{key}: {value}")

In [ ]:
cd = {c['entity'].id: c for c in chats}

In [ ]:
from get_chat_list import categorize_entity


def get_chat_info(chat_id):
    res = {}
    try:
        res['username'] = cd[chat_id]['entity'].username
    except:
        res['username'] = None
    try:
        res['title'] = cd[chat_id]['entity'].title
    except:
        res['title'] = None
    res['type'] = categorize_entity(cd[chat_id]['entity'])
    return res


get_chat_info(4285323792)

In [ ]:

message = messages[768142633][0]

for attr in dir(message):
    if not attr.startswith('_'):  # Skip private attributes
        print(f"{attr}: {getattr(message, attr)}")




In [ ]:
print(message.stringify())

In [ ]:
print(message.to_json(indent=2, ensure_ascii=False))


In [ ]:
c1 = cd[4285323792]['entity']
print(c1.to_json(indent=2, ensure_ascii=False))


In [ ]:
from collections import defaultdict


def calculate_stats(messages):
    # a) total messages
    # b) peak messages / day
    # c) peak messages / month
    # d) total messages last year
    # e) chats with media

    stats = {}

    for chat_id, m in messages.items():
        s = {}
        s['total_messages'] = len(m)
        s['media_count'] = sum(1 for message in m if message.media)

        # calculate peak messages / day
        messages_counts_by_day = defaultdict(int)
        for message in m:
            messages_counts_by_day[message.date.date()] += 1
        s['peak_messages_per_day'] = max(messages_counts_by_day.values())

        # calculate peak messages / month
        messages_counts_by_month = defaultdict(int)
        for message in m:
            messages_counts_by_month[message.date.month] += 1
        s['peak_messages_per_month'] = max(messages_counts_by_month.values())

        # calculate total messages last year
        s['total_messages_last_year'] = sum(
            1 for message in m if message.date.year == datetime.now().year - 1)

        stats[chat_id] = s
    return stats

In [ ]:
stats = calculate_stats(messages)

In [ ]:
# let's plot stats
from matplotlib import pyplot as plt
import seaborn as sns

# sns.set_style("whitegrid")
plt.style.use('dark_background')
# sns.set_style("darkgrid")
# sns.set_palette("husl")
# sns.set_palette("pastel")
sns.set_palette("deep")  # Using deep palette which shows better on dark background


def plot_stats(stats):
    for key in [
        'total_messages',
        'peak_messages_per_day',
        'peak_messages_per_month',
        'total_messages_last_year',
        'media_count'
    ]:
        # Create histogram
        plt.figure(figsize=(10, 6))
        plt.hist([s[key] for s in stats.values()], bins=20)
        plt.title(f"Distribution of {key}")
        plt.xlabel(key)
        plt.ylabel("Count")
        plt.show()


plot_stats(stats)





In [ ]:
# let's save and load messages

def save_messages(messages, path="data/messages.json"):
    data = {
        k: [v.to_json(indent=2, ensure_ascii=False) for v in v]
        for k, v in messages.items()
    }
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


from telethon.tl.types import Message, MessageService

type_map = {
    'Message': lambda x: Message(**x),
    'MessageService': lambda x: MessageService(**x),
}


def load_messages(path="data/messages.json"):
    messages = {}
    error_count = 0
    with open(path, "r") as f:
        data = json.load(f)
        for k, v in data.items():
            messages[k] = []
            for vv in v:
                d = json.loads(vv)
                data_type = type_map[d.pop("_")]
                try:
                    messages[k].append(data_type(d))
                except Exception as e:
                    messages[k].append(d)
                    error_count += 1

    logger.info(f"Error count: {error_count}")
    return messages


save_messages(messages)
mm = load_messages()




In [ ]:
mm